In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from glam import Geocoder, download_dependencies
from pathlib import Path

import requests
import geopandas as gpd

## Checkpoint: load cached processed data if available

Skips straight to a df with addresses parsed, geocoded, and hazard-flagged —
avoids re-running the ~30min geocoding pass. If this loads successfully, skip
ahead to the "Analysis" section below rather than re-running the parsing /
geocoding / hazard cells.

In [10]:
CACHE_PATH = Path("./cache/dvrs_processed.parquet")

if CACHE_PATH.exists():
    df = pd.read_parquet(CACHE_PATH)
    print(f"Loaded cached df from {CACHE_PATH}, shape={df.shape}")
else:
    df = None
    print("No cache found — run the full pipeline below, then save a checkpoint at the end.")


Loaded cached df from cache\dvrs_processed.parquet, shape=(92010, 48)


In [3]:
import csv
import io

with open("DVRS270826.txt", encoding="utf-16") as f:
    lines = f.read().splitlines()

header, *rows = lines
expected_cols = len(header.split("|"))


def unwrap(line):
    if len(line) >= 2 and line[0] == '"' and line[-1] == '"':
        return line[1:-1]
    return line


cleaned = [unwrap(line) for line in rows]

# Drop genuinely malformed rows (e.g. a stray "#NAME?" Excel-error row) instead
# of letting them silently misalign columns.
good_rows = [line for line in cleaned if len(line.split("|")) == expected_cols]
n_dropped = len(cleaned) - len(good_rows)
if n_dropped:
    print(f"Dropping {n_dropped} malformed row(s) that don't split into {expected_cols} fields")

csv_text = "\n".join([header] + good_rows)
df = pd.read_csv(io.StringIO(csv_text), sep="|", dtype=str, quoting=csv.QUOTE_NONE)


Dropping 1 malformed row(s) that don't split into 41 fields


In [4]:
# Normalize column names

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 92010 entries, 0 to 92009
Data columns (total 41 columns):
 #   Column                            Non-Null Count  Dtype
---  ------                            --------------  -----
 0   valuation_roll_number             92010 non-null  str  
 1   valuation_number_assessment       92010 non-null  str  
 2   valuation_number_suffix           14153 non-null  str  
 3   sale_date                         92010 non-null  str  
 4   district_code                     92010 non-null  str  
 5   sale_type                         92009 non-null  str  
 6   sales_group                       92010 non-null  str  
 7   sale_tenure                       92010 non-null  str  
 8   price_value_relationship          92010 non-null  str  
 9   sale_price_gross                  92010 non-null  str  
 10  sale_price_net                    92010 non-null  str  
 11  sale_price_chattels               92010 non-null  str  
 12  sale_price_other                  92010 non

In [6]:
# Sanity check
print(df.shape)
print(df[["sale_date", "sale_price_gross", "sale_price_net"]].isna().sum())

(92010, 41)
sale_date           0
sale_price_gross    0
sale_price_net      0
dtype: int64


## Geocode via LINZ Address Matching

In [7]:
def download_glam_dependencies(deps_directory: str) -> Path:
    deps_path = Path(deps_directory)
    marker = deps_path / "nz-street-address.csv"

    if marker.exists():
        print(f"glam dependencies already present in {deps_path}, skipping download")
    else:
        download_dependencies(str(deps_path))


In [8]:
deps_dir = Path("./glam-deps")
if not (deps_dir / "nz-street-address.csv").exists():
    download_glam_dependencies(str(deps_dir))

gc = Geocoder("./glam-deps", matcher="tfidf", parser="rnn")


19:07 - GLAM - INFO - Loading NZSA data from glam-deps\nz-street-address.csv
19:07 - GLAM - INFO - Using existing postcodes


In [9]:
# Subsetting df for experimental
# df = df[:10000]

In [10]:
# Build a search address per row from what we have (no suburb available yet).
addresses = (
    df["situation_number"].fillna("") + " " + df["situation_name"].fillna("")
).str.strip()

results = gc.geocode_addresses(addresses.tolist())

df["geocode_confidence"] = [r.confidence for r in results]
df["longitude"] = [r.matched_address.shape_X if r.matched_address else None for r in results]
df["latitude"] = [r.matched_address.shape_Y if r.matched_address else None for r in results]
df["matched_suburb"] = [r.matched_address.suburb_locality if r.matched_address else None for r in results]
df["matched_postcode"] = [r.matched_address.postcode if r.matched_address else None for r in results]

print(df["geocode_confidence"].describe())
print("no match at all:", df["longitude"].isna().sum())


19:07 - GLAM - INFO - Loading dependencies for TFIDF parser
d:\Codes\COMPSCI_760_Group_3_Intelligent_Real_Estate_Price_Prediction_Model\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Codes\COMPSCI_760_Group_3_Intelligent_Real_Estate_Price_Prediction_Model\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
19:07 - GLAM - INFO - Beginning

count    92010.000000
mean         0.649989
std          0.112107
min          0.000000
25%          0.600440
50%          0.652555
75%          0.707016
max          0.919454
Name: geocode_confidence, dtype: float64
no match at all: 0


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 92010 entries, 0 to 92009
Data columns (total 48 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   valuation_roll_number             92010 non-null  str    
 1   valuation_number_assessment       92010 non-null  str    
 2   valuation_number_suffix           14153 non-null  str    
 3   sale_date                         92010 non-null  str    
 4   district_code                     92010 non-null  str    
 5   sale_type                         92009 non-null  str    
 6   sales_group                       92010 non-null  str    
 7   sale_tenure                       92010 non-null  str    
 8   price_value_relationship          92010 non-null  str    
 9   sale_price_gross                  92010 non-null  str    
 10  sale_price_net                    92010 non-null  str    
 11  sale_price_chattels               92010 non-null  str    
 12  sale_price_othe

In [6]:
flood_plains = gpd.read_file("./hazard_data/flood_plains.geojson")
flood_sensitive = gpd.read_file("./hazard_data/flood_sensitive_area.geojson")
unitary_plan_zones = gpd.read_file("./hazard_data/Unitary_Plan_Base_Zone.geojson")


In [15]:
geocoded = df.dropna(subset=["longitude", "latitude"])
point_gdf = gpd.GeoDataFrame(
    geocoded[[]],
    geometry = gpd.points_from_xy(geocoded["longitude"], geocoded["latitude"]),
    crs="EPSG:4326",
)



In [16]:
flood_plains_matches = gpd.sjoin(point_gdf, flood_plains, how = "inner", predicate="within")
df["in_flood_plain"] = df.index.isin(flood_plains_matches.index)

In [17]:

flood_sensitive_matches = gpd.sjoin(point_gdf, flood_sensitive, how = "inner", predicate="within")
df["in_flood_sensitive_area"] = df.index.isin(flood_sensitive_matches.index)

In [19]:
unitary_zone = gpd.sjoin(point_gdf, unitary_plan_zones, how="left", predicate="within")
unitary_zone = unitary_zone[~unitary_zone.index.duplicated(keep="first")]  # in case of overlapping zone polygons
df["unitary_plan_zone"] = unitary_zone["ZONE"]

In [22]:
df["in_flood_plain"].describe()

count     92010
unique        2
top       False
freq      87473
Name: in_flood_plain, dtype: object

In [24]:
df["in_flood_sensitive_area"].describe()

count     92010
unique        2
top       False
freq      90526
Name: in_flood_sensitive_area, dtype: object

In [26]:
df["unitary_plan_zone"].describe()

count    57139.000000
mean        26.400672
std         18.925131
min          1.000000
25%         18.000000
50%         18.000000
75%         32.000000
max         69.000000
Name: unitary_plan_zone, dtype: float64

In [32]:
df["unitary_plan_zone"].unique()

array([60., nan, 18.,  8.,  5., 20., 19., 12., 16., 44., 69., 17., 22.,
       49., 33., 32.,  7., 10., 46., 51., 11., 35., 63., 34.,  4., 30.,
       27., 40.,  3., 23., 31.,  1., 54., 15., 43., 53., 68., 56., 55.,
       61., 52.])

In [33]:
points_nztm = point_gdf.to_crs("EPSG:2193")
flood_plains_nztm = flood_plains.to_crs("EPSG:2193")
flood_sensitive_nztm = flood_sensitive.to_crs("EPSG:2193")

In [34]:
flood_plains_nearest = gpd.sjoin_nearest(
    points_nztm, flood_plains_nztm, distance_col="dist_to_flood_plain_m"
)
flood_plains_nearest = flood_plains_nearest[~flood_plains_nearest.index.duplicated(keep="first")]
df["dist_to_flood_plain_m"] = flood_plains_nearest["dist_to_flood_plain_m"]

flood_sensitive_nearest = gpd.sjoin_nearest(
    points_nztm, flood_sensitive_nztm, distance_col="dist_to_flood_sensitive_area_m"
)
flood_sensitive_nearest = flood_sensitive_nearest[~flood_sensitive_nearest.index.duplicated(keep="first")]
df["dist_to_flood_sensitive_area_m"] = flood_sensitive_nearest["dist_to_flood_sensitive_area_m"]

print(df[["dist_to_flood_plain_m", "dist_to_flood_sensitive_area_m"]].describe())

       dist_to_flood_plain_m  dist_to_flood_sensitive_area_m
count           9.201000e+04                    9.201000e+04
mean            1.501222e+05                    1.575913e+05
std             2.827587e+05                    2.854218e+05
min             0.000000e+00                    0.000000e+00
25%             5.443890e+01                    7.896345e+02
50%             1.565186e+02                    3.551495e+03
75%             1.481008e+05                    1.685662e+05
max             1.158145e+06                    1.172344e+06


In [35]:
df[["dist_to_flood_plain_m", "dist_to_flood_sensitive_area_m"]].head(5)

,dist_to_flood_plain_m,dist_to_flood_sensitive_area_m
0,2.419102e+02,4.866407e+03
1,1.018463e+06,1.031195e+06
2,1.962920e+02,4.670897e+03
3,1.882070e+01,4.814777e+03
4,6.764028e+00,4.804595e+03


## Flag properties inside Auckland Council flood plain hazard zones

In [27]:
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(CACHE_PATH)
print(f"Saved checkpoint to {CACHE_PATH}, shape={df.shape}")


Saved checkpoint to cache\dvrs_processed.parquet, shape=(92010, 49)


## Analysis